# **기초인공지능 프로젝트 - 자차속도추정 모델 설계**

## train.zip 파일과 test.zip 파일 / train 파일의 GT Speed 파일과 test 파일의 GT Speed 파일 업로드

### 프로젝트 시작 최초 한 번만 업로드 후 다음 셀에 있는 압축 해제 코드 실행하면 다시 업로드 안해도 됩니다.



In [ ]:
from google.colab import drive
from google.colab import files
import os
import zipfile

# Google Drive 마운트
drive.mount('/content/drive')

# Google Drive 저장 경로 설정
drive_data_path = '/content/drive/MyDrive/Colab_Data'
os.makedirs(drive_data_path, exist_ok=True)

# 필요한 파일 업로드
print("train.zip 파일을 업로드하세요:")
uploaded_train_zip = files.upload()

print("test.zip 파일을 업로드하세요:")
uploaded_test_zip = files.upload()

print("train_speeds.txt 파일을 업로드하세요:")
uploaded_train_speeds = files.upload()

print("test_speeds.txt 파일을 업로드하세요:")
uploaded_test_speeds = files.upload()

# 업로드된 파일을 Google Drive에 저장
for filename in uploaded_train_zip.keys():
    save_path = os.path.join(drive_data_path, filename)
    with open(save_path, 'wb') as f:
        f.write(uploaded_train_zip[filename])
    print(f"train.zip이 Google Drive에 저장되었습니다: {save_path}")

for filename in uploaded_test_zip.keys():
    save_path = os.path.join(drive_data_path, filename)
    with open(save_path, 'wb') as f:
        f.write(uploaded_test_zip[filename])
    print(f"test.zip이 Google Drive에 저장되었습니다: {save_path}")

for filename in uploaded_train_speeds.keys():
    save_path = os.path.join(drive_data_path, filename)
    with open(save_path, 'wb') as f:
        f.write(uploaded_train_speeds[filename])
    print(f"train_speeds.txt가 Google Drive에 저장되었습니다: {save_path}")

for filename in uploaded_test_speeds.keys():
    save_path = os.path.join(drive_data_path, filename)
    with open(save_path, 'wb') as f:
        f.write(uploaded_test_speeds[filename])
    print(f"test_speeds.txt가 Google Drive에 저장되었습니다: {save_path}")


Mounted at /content/drive
train.zip 파일을 업로드하세요:


Saving train.zip to train.zip
test.zip 파일을 업로드하세요:


Saving test.zip to test.zip
train_speeds.txt 파일을 업로드하세요:


Saving train_speeds.txt to train_speeds.txt
test_speeds.txt 파일을 업로드하세요:


Saving test_speeds.txt to test_speeds.txt
train.zip이 Google Drive에 저장되었습니다: /content/drive/MyDrive/Colab_Data/train.zip
test.zip이 Google Drive에 저장되었습니다: /content/drive/MyDrive/Colab_Data/test.zip
train_speeds.txt가 Google Drive에 저장되었습니다: /content/drive/MyDrive/Colab_Data/train_speeds.txt
test_speeds.txt가 Google Drive에 저장되었습니다: /content/drive/MyDrive/Colab_Data/test_speeds.txt


## 업로드한 zip 파일을 압축 해제 (위에서 Google Drive에 한 번 업로드 한 뒤에는 이 코드부터 실행)

In [8]:
from google.colab import drive
from google.colab import files
import os
import zipfile

# Google Drive 마운트
drive.mount('/content/drive')

# Google Drive에 저장된 파일 경로
train_zip_path = '/content/drive/MyDrive/Colab_Data/train.zip'
test_zip_path = '/content/drive/MyDrive/Colab_Data/test.zip'
train_speed_file = '/content/drive/MyDrive/Colab_Data/train_speeds.txt'
test_speed_file = '/content/drive/MyDrive/Colab_Data/test_speeds.txt'

# 압축 해제 경로 설정
train_dir = '/content/train'
test_dir = '/content/test'
os.makedirs(train_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

# ZIP 파일 압축 해제
with zipfile.ZipFile(train_zip_path, 'r') as zip_ref:
    zip_ref.extractall(train_dir)
with zipfile.ZipFile(test_zip_path, 'r') as zip_ref:
    zip_ref.extractall(test_dir)

print("Train 및 Test 데이터가 성공적으로 압축 해제되었습니다.")
print(f"Train 데이터 경로: {train_dir}")
print(f"Test 데이터 경로: {test_dir}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Train 및 Test 데이터가 성공적으로 압축 해제되었습니다.
Train 데이터 경로: /content/train
Test 데이터 경로: /content/test


## Data Loader

In [9]:
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
import os
import torch

class SpeedDataset(Dataset):
    def __init__(self, image_dir, speed_file, sequence_length=6, transform=None):
        self.image_dir = image_dir
        self.speed_file = speed_file
        self.sequence_length = sequence_length
        self.transform = transform

        # 디버깅: 데이터 경로 및 파일 리스트 확인
        print("Image directory:", self.image_dir)
        print("Image files:", sorted(os.listdir(self.image_dir)))

        # 이미지 파일 정렬 및 필터링
        self.image_files = sorted([f for f in os.listdir(self.image_dir) if f.endswith('.png')])

        if len(self.image_files) < self.sequence_length:
            raise ValueError("Sequence length is greater than the number of image files.")

        # 속도 파일 로드
        with open(self.speed_file, 'r') as f:
            self.speeds = [float(line.strip()) for line in f.readlines()]

        if len(self.speeds) < len(self.image_files):
            raise ValueError("Speed file does not match the number of images.")

    def __len__(self):
        return len(self.image_files) - self.sequence_length + 1

    def __getitem__(self, idx):
        image_sequence = []
        for i in range(self.sequence_length):
            img_path = os.path.join(self.image_dir, self.image_files[idx + i])
            img = Image.open(img_path).convert('RGB')
            if self.transform:
                img = self.transform(img)
            image_sequence.append(img)
        images = torch.stack(image_sequence).permute(1, 0, 2, 3)
        speed = torch.tensor(self.speeds[idx + self.sequence_length - 1], dtype=torch.float32)

        # 6장의 연속된 이미지들과 6장 중 마지막 이미지의 speed 값 Return
        return images, speed


## Transform 설정 및 Loader 선언

In [23]:
from torch.utils.data import DataLoader
import numpy as np

def calculate_mean_std(dataset):
    loader = DataLoader(dataset, batch_size=64, shuffle=False)
    mean = 0.0
    std = 0.0
    total_images = 0
    for images, _ in loader:
        batch_samples = images.size(0)  # Batch size
        images = images.view(batch_samples, images.size(1), -1)
        mean += images.mean(2).sum(0)
        std += images.std(2).sum(0)
        total_images += batch_samples
    mean /= total_images
    std /= total_images
    return mean, std

# 예시 데이터셋에서 평균/표준편차 계산
mean, std = calculate_mean_std(train_dataset)
print(f"Mean: {mean}, Std: {std}")


Mean: tensor([-0.2755, -0.2627, -0.2878]), Std: tensor([0.6952, 0.6978, 0.6973])


In [44]:
# train을 위한 transform --> 수정 가능
# 사이즈를 400,121로 설정. 모델 정확도가 떨어진다면 바꾸려햇지만 바꾸지 않음.
transform_for_train = transforms.Compose([
    transforms.Resize((400, 121)),
    transforms.ToTensor(),

    transforms.Normalize(mean=[-0.2755, -0.2627, -0.2878], std=[0.6952, 0.6978, 0.6973])
])

# test를 위한 transform 은 수정하지 말 것. --> 수정 시 0점.
# 정규화만 시켰음
transform_for_test = transforms.Compose([
    transforms.Resize((400, 121)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[-0.2755, -0.2627, -0.2878], std=[0.6952, 0.6978, 0.6973])
])

train_dir = '/content/train/train'
test_dir = '/content/test/test'

# 데이터셋 생성
train_dataset = SpeedDataset(train_dir, train_speed_file, transform=transform_for_train)
test_dataset = SpeedDataset(test_dir, test_speed_file, transform=transform_for_test)

batch_size_train = 16 # train을 위한 batch size --> 수정 가능
batch_size_test = 16 # test를 위한 batch size 수정하지 말 것. --> 수정 시 0점.

# DataLoader 생성
# test 로더는 성능 체크를 위해 shuffle을 Flase로 해놓음. train 데이터는 True해야 더 잘 모델 학습이 됨
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size_train, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size_test, shuffle=False)


Image directory: /content/train/train
Image files: ['0000000000.png', '0000000001.png', '0000000002.png', '0000000003.png', '0000000004.png', '0000000005.png', '0000000006.png', '0000000007.png', '0000000008.png', '0000000009.png', '0000000010.png', '0000000011.png', '0000000012.png', '0000000013.png', '0000000014.png', '0000000015.png', '0000000016.png', '0000000017.png', '0000000018.png', '0000000019.png', '0000000020.png', '0000000021.png', '0000000022.png', '0000000023.png', '0000000024.png', '0000000025.png', '0000000026.png', '0000000027.png', '0000000028.png', '0000000029.png', '0000000030.png', '0000000031.png', '0000000032.png', '0000000033.png', '0000000034.png', '0000000035.png', '0000000036.png', '0000000037.png', '0000000038.png', '0000000039.png', '0000000040.png', '0000000041.png', '0000000042.png', '0000000043.png', '0000000044.png', '0000000045.png', '0000000046.png', '0000000047.png', '0000000048.png', '0000000049.png', '0000000050.png', '0000000051.png', '0000000052.

In [45]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import torch.nn as nn

class SpeedPredictionModel(nn.Module):
    def __init__(self):
        super(SpeedPredictionModel, self).__init__()
        # CNN과 LSTM 구조는 동일
        self.conv1 = nn.Conv3d(3, 16, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
        self.bn1 = nn.BatchNorm3d(16)
        self.conv2 = nn.Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1))
        self.bn2 = nn.BatchNorm3d(32)
        self.conv3 = nn.Conv3d(32, 64, kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1))
        self.bn3 = nn.BatchNorm3d(64)

        self.lstm = nn.LSTM(input_size=64, hidden_size=128, num_layers=1, batch_first=True)
        self.dropout = nn.Dropout(p=0.5)  # Dropout 추가
        self.fc1 = nn.Linear(128, 256)
        self.fc2 = nn.Linear(256, 1)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))

        x = x.view(x.size(0), -1, 64)
        x, _ = self.lstm(x)

        x = F.relu(self.fc1(x[:, -1, :]))
        x = self.dropout(x)  # Dropout 적용
        speed = self.fc2(x)
        return speed







## Model 선언 및 loss function, optimizer, epochs 설정

In [46]:
# 모델 생성
# model = SpeedPredictionModel().cuda()  # GPU 사용


model = SpeedPredictionModel().cuda()


# 손실 함수 및 옵티마이저
criterion_for_train = nn.L1Loss()  # 모델에 맞는 loss function 설정

criterion_for_test = nn.L1Loss()  # Test 과정에서는 Mean Absolute Error 사용. 수정 시 0점.
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)  # 모델에 맞는 optimizer 설정 // 가중치 감소에 더 효과적인 AdamW를 사용
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)
epochs = 20  # 원하는 epochs 수 설정




## Train 및 Test 코드 (수정하지 말 것 --> 수정 시 0점)

In [47]:
def train_and_evaluate(model, train_loader, test_loader, optimizer, scheduler=None, num_epochs=20):
    best_test_loss = float('inf')  # 최적의 test loss 값 저장
    best_epoch = 0  # 최적의 테스트 손실이 발생한 epoch

    for epoch in range(num_epochs):
        # 학습 단계
        model.train()
        running_loss = 0.0
        for images, speeds in train_loader:
            images, speeds = images.cuda(), speeds.cuda()

            # Forward Pass
            outputs = model(images)
            loss = criterion_for_train(outputs.squeeze(), speeds)

            # Backward Pass and Optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        train_loss = running_loss / len(train_loader)

        # 평가 단계
        model.eval()
        test_loss = 0.0
        all_predictions = []  # 모든 예측값 저장
        all_actuals = []  # 모든 실제값 저장
        with torch.no_grad():
            for images, speeds in test_loader:
                images, speeds = images.cuda(), speeds.cuda()
                outputs = model(images)
                loss = criterion_for_test(outputs.squeeze(), speeds)
                test_loss += loss.item()

                # 예측값 및 실제값 저장
                all_predictions.extend(outputs.squeeze().cpu().tolist())
                all_actuals.extend(speeds.cpu().tolist())
        test_loss /= len(test_loader)

        # 최적의 테스트 손실 업데이트
        # 최적의 테스트 손실 갱신 시 예측/실제값 출력
        if test_loss < best_test_loss:
            best_test_loss = test_loss
            best_epoch = epoch + 1

            print(f"\nNew Best Test Loss: {best_test_loss:.4f} at Epoch {best_epoch}")
            print("Predictions vs Actuals:")
            for i, (pred, actual) in enumerate(zip(all_predictions, all_actuals)):
                print(f"Index {i}: Predicted: {pred:.2f}, Actual: {actual:.2f}")

        print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}")

        # Scheduler 업데이트 (epoch 종료 후)
        if scheduler is not None:
            scheduler.step()
            print(f"Updated Learning Rate: {scheduler.get_last_lr()}")

    # 최적의 테스트 손실 출력
    print(f"Best Test Loss: {best_test_loss:.4f} at Epoch {best_epoch}")


In [48]:
train_and_evaluate(model, train_loader, test_loader, optimizer,scheduler, num_epochs=epochs)


New Best Test Loss: 7.6996 at Epoch 1
Predictions vs Actuals:
Index 0: Predicted: 1.40, Actual: 12.94
Index 1: Predicted: 1.40, Actual: 12.88
Index 2: Predicted: 1.38, Actual: 12.81
Index 3: Predicted: 1.38, Actual: 12.73
Index 4: Predicted: 1.38, Actual: 12.66
Index 5: Predicted: 1.38, Actual: 12.58
Index 6: Predicted: 1.38, Actual: 12.48
Index 7: Predicted: 1.38, Actual: 12.37
Index 8: Predicted: 1.38, Actual: 12.27
Index 9: Predicted: 1.37, Actual: 12.15
Index 10: Predicted: 1.37, Actual: 12.07
Index 11: Predicted: 1.37, Actual: 12.00
Index 12: Predicted: 1.37, Actual: 11.95
Index 13: Predicted: 1.38, Actual: 11.91
Index 14: Predicted: 1.37, Actual: 11.87
Index 15: Predicted: 1.38, Actual: 11.83
Index 16: Predicted: 1.38, Actual: 11.79
Index 17: Predicted: 1.39, Actual: 11.75
Index 18: Predicted: 1.40, Actual: 11.71
Index 19: Predicted: 1.41, Actual: 11.68
Index 20: Predicted: 1.42, Actual: 11.65
Index 21: Predicted: 1.44, Actual: 11.62
Index 22: Predicted: 1.45, Actual: 11.59
Inde